In [1]:
from mirtorch.linear import Identity, Diff3dgram, NuSense, NuSenseGram
from mirtorch.prox import Const, L1Regularizer
from obj.RawData import RawData
from obj.FSLViewer import FSLViewer
from obj.Smap import Smaps
from obj.pblms import PBNuSense, PBNuSenseGram
from obj.helpers import *
from copy import deepcopy
import torch

import matplotlib.pyplot as plt

In [2]:
# Flags
disp = False     # Display Images?
save = True     # Save Images?

# Check for the best CUDA device
device = best_available_device(verbose=True)
torch.manual_seed(0)

# Instantiate FSLViewer
viewer = FSLViewer(halted=disp)

Using cuda


In [3]:
# Load Data
y = RawData.from_dir("/home/clouly/DEV/asldata/20261104/asldata_*smap*")

In [4]:
y.data.shape

torch.Size([708, 360, 1, 32])

In [5]:
y.traj.shape

torch.Size([708, 30, 12, 3])

In [6]:
y.traj = y.traj.swapdims(1, 2).flatten(start_dim=1, end_dim=2).unsqueeze(2)
y.traj.shape

torch.Size([708, 360, 1, 3])

In [7]:
# We:
#   Clip off the first 50 kspace samples
#   Reorganize so that we are reconstructing coil images
#   Reshape the data 
y = y.initial_clip().coilwise().set_mirt_shape()

In [8]:
w = y.get_dcf(device=device)

In [10]:
s = torch.ones((1,)+tuple(y.matsize), device=y.device, dtype=y._data_type)
s = Smaps._from_tensors(s, None)

In [11]:
y.traj.shape

torch.Size([1, 3, 236880])

In [12]:
y.traj = y.traj.expand(32, -1, -1)

In [13]:
s.shape

torch.Size([1, 64, 64, 64])

In [14]:
img = y.recon_CGSENSE(s, 1e-1, w, device=device, batchmode = True)

In [21]:
viewer.unhalt()

In [15]:
viewer.add_complex(img[0], tdim=0, name="img_r1en1_")

In [16]:
from sigpy.mri.app import EspiritCalib
import numpy as np
from sigpy import cpu_device

/home/clouly/.venvs/global/lib/python3.12/site-packages/sigpy/config.py:27: UserWarning: Importing cupy.cuda.cudnn failed. For more details, see the error stack below:
libcudnn.so.8: cannot open shared object file: No such file or directory
  warnings.warn(
/home/clouly/.venvs/global/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
calib_width = 20
kernel_width = 5
thresh = 0.02
crop = 0.95

In [19]:
img = img[0]

In [64]:
kspace = torch.fft.fftshift(
    torch.fft.fftn(
        torch.fft.ifftshift(img, dim=(-3, -2, -1)),
        dim=(-3, -2, -1),
        norm="ortho",
    ),
    dim=(-3, -2, -1),
)

In [65]:
device = kspace.device
kspace_np = kspace.detach().cpu().numpy().astype(np.complex64)

# ESPIRiT calibration.
app = EspiritCalib(
    kspace_np,
    calib_width=calib_width,
    kernel_width=kernel_width,
    thresh=thresh,
    crop=crop,
    max_iter=100,
    device=cpu_device,
    show_pbar=True,
)

smaps_np = app.run()

# Return to PyTorch.
smaps = torch.from_numpy(smaps_np.copy()).to(
    device=device,
    dtype=kspace.dtype,
)

EspiritCalib: 100%|██████████| 100/100 [00:21<00:00,  4.60it/s]


In [28]:
viewer.add_complex(smaps[0], tdim=0, name="smap")

In [99]:
viewer.add_complex(torch.abs(smaps[1]).diff(dim=0).diff(dim=1).diff(dim=2), name="eig_r1en3")

In [27]:
viewer.set_save_dir("/home/clouly/DEV/asldata/20261104/asldata_smap")

In [23]:

import torch
import torch.nn.functional as F


def espirit(
    coil_images,
    calib_width=20,
    kernel_width=5,
    thresh=0.02,
    crop=0.95,
    max_iter=50,
    kernel_batch=4,
):
    """
    GPU ESPIRiT calibration from reconstructed coil images.

    Parameters
    ----------
    coil_images : complex Tensor
        [ncoils, nx, ny, nz]

    Returns
    -------
    smaps : complex Tensor
        [ncoils, nx, ny, nz]
    eigenvalue : real Tensor
        [nx, ny, nz]
    """

    device = coil_images.device
    nc, nx, ny, nz = coil_images.shape
    k = kernel_width
    dims = (-3, -2, -1)

    with torch.no_grad():

        # --------------------------------------------------
        # 1. Coil images -> centered Cartesian k-space
        # --------------------------------------------------

        kspace = torch.fft.fftshift(
            torch.fft.fftn(
                torch.fft.ifftshift(coil_images, dim=dims),
                dim=dims,
                norm="ortho",
            ),
            dim=dims,
        )

        # --------------------------------------------------
        # 2. Extract calibration region
        # --------------------------------------------------

        starts = [
            (n - calib_width) // 2
            for n in (nx, ny, nz)
        ]

        calib = kspace[
            :,
            starts[0]:starts[0] + calib_width,
            starts[1]:starts[1] + calib_width,
            starts[2]:starts[2] + calib_width,
        ]

        # --------------------------------------------------
        # 3. Calibration matrix
        # --------------------------------------------------

        patches = (
            calib
            .unfold(1, k, 1)
            .unfold(2, k, 1)
            .unfold(3, k, 1)
        )

        A = patches.permute(
            1, 2, 3, 0, 4, 5, 6
        ).reshape(-1, nc * k**3)

        # --------------------------------------------------
        # 4. SVD and kernel selection
        # --------------------------------------------------

        _, S, Vh = torch.linalg.svd(
            A, full_matrices=False
        )

        kernels = Vh[S > thresh * S[0]].reshape(
            -1, nc, k, k, k
        )

        # --------------------------------------------------
        # 5. Construct ESPIRiT covariance matrix
        # --------------------------------------------------

        # Shape: [nx, ny, nz, nc, nc]
        AHA = torch.zeros(
            nx, ny, nz, nc, nc,
            dtype=coil_images.dtype,
            device=device,
        )

        pad = []
        for n in (nz, ny, nx):
            left = (n - k) // 2
            right = n - k - left
            pad.extend([left, right])

        for batch in kernels.split(kernel_batch):

            # Center-pad kernels to image dimensions.
            padded = F.pad(batch, pad)

            # Centered inverse FFT.
            img_kernels = torch.fft.fftshift(
                torch.fft.ifftn(
                    torch.fft.ifftshift(padded, dim=dims),
                    dim=dims,
                    norm="ortho",
                ),
                dim=dims,
            )

            # Sum outer products across calibration kernels.
            AHA += torch.einsum(
                "q c x y z, q d x y z -> x y z c d",
                img_kernels,
                img_kernels.conj(),
            )

        AHA *= (nx * ny * nz) / k**3

        # --------------------------------------------------
        # 6. Principal eigenvector via power iteration
        # --------------------------------------------------

        v = torch.ones(
            nx, ny, nz, nc,
            dtype=coil_images.dtype,
            device=device,
        )

        v = F.normalize(v, dim=-1)

        for _ in range(max_iter):
            v = torch.einsum(
                "...ij,...j->...i", AHA, v
            )
            v = F.normalize(v, dim=-1)

        # Maximum eigenvalue (Rayleigh quotient).
        Av = torch.einsum(
            "...ij,...j->...i", AHA, v
        )

        eigenvalue = (
            v.conj() * Av
        ).sum(dim=-1).real

        # --------------------------------------------------
        # 7. Phase normalization and eigenvalue cropping
        # --------------------------------------------------

        smaps = v.permute(3, 0, 1, 2)

        phase = smaps[0].conj() / (
            smaps[0].abs() + 1e-8
        )

        smaps = smaps * phase.unsqueeze(0)

        smaps = smaps * (eigenvalue > crop).unsqueeze(0)

        return smaps, eigenvalue


In [24]:
smaps = espirit(img.to(device), calib_width, kernel_width, thresh, 0., max_iter=100)

In [104]:
del viewer